In [1]:
# NOTEBOOK 21 - does the wind graph feed information in the right direction?
# In 03_dynamic_adjacency, A[i,j] > 0 means the wind AT station i blows TOWARD station j (i is upwind of j).
# The model (09_model_v2) then does  h_i_new = sum_j A[i,j] * h_j , i.e. station i gathers from the
# stations DOWNWIND of it. A receiver should gather from its UPWIND sources, which is A transposed.
# This notebook trains the same 09 model with A and A_fc transposed and compares it with the saved v2 models.
# Nothing existing is overwritten: new checkpoints are saved as watgcn_v2T_seed*.pt.
# HOW TO CHECK: this cell only prints a note.
print('Direction test: A as built (rows = upwind source) vs A transposed (rows = receiver).')

Direction test: A as built (rows = upwind source) vs A transposed (rows = receiver).


In [2]:
import os, json, time, copy, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

BASE   = r'D:\FYP\Eco-Twin'
V2_DIR = os.path.join(BASE, 'v2_clean')
OUT    = os.path.join(V2_DIR, 'ecotwin_outputs')
SCALER_DIR, PLOT_DIR   = os.path.join(OUT, 'scalers'), os.path.join(OUT, 'plots')
MODEL_DIR, RESULTS_DIR = os.path.join(OUT, 'models'),  os.path.join(OUT, 'results')

DATASET_CSV      = os.path.join(V2_DIR, 'EcoTwin_Final_Dataset.csv')
DYNAMIC_ADJ_PATH = os.path.join(V2_DIR, 'dynamic_adj_matrix.npy')
STATIONS_EXCEL   = os.path.join(V2_DIR, 'Delhi_list_v2.xlsx')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

INPUT_WINDOW, HORIZON = 48, 24
BATCH_SIZE = 16
HIDDEN_DIM, GRU_LAYERS, ATTN_HEADS, DROPOUT = 32, 1, 4, 0.2
GCN_HOPS       = 2
LOSS_VOL_ALPHA = 1.0                       # volatility up-weighting (0 = plain Huber)
SEEDS          = [42, 7, 123, 2024, 999, 2025, 77]
THRESHOLD_KM   = 12.0
V2_TAG         = 'watgcn_v2'               # 07's files are never overwritten

print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU   : {torch.cuda.get_device_name(0)}')
print(f'Seeds : {SEEDS}')

# HOW TO CHECK: should print 'cuda' + your RTX 3050. 'cpu' still works, ~10x slower.

Device: cuda
GPU   : NVIDIA GeForce RTX 3050 6GB Laptop GPU
Seeds : [42, 7, 123, 2024, 999, 2025, 77]


In [3]:
df = pd.read_csv(DATASET_CSV, parse_dates=['Timestamp'])
df['Station'] = df['Station'].str.strip()
df = df.sort_values(['Station', 'Timestamp']).reset_index(drop=True)

df_st = pd.read_excel(STATIONS_EXCEL)
df_st.columns = df_st.columns.str.strip().str.lower()
df_st['station'] = df_st['station'].str.strip()
station_names = df_st['station'].tolist()
lats = df_st['latitude'].values.astype(float)
lons = df_st['longitude'].values.astype(float)
N = len(station_names)

timestamps = np.array(sorted(df['Timestamp'].unique()))
T = len(timestamps)
dyn_adj = np.load(DYNAMIC_ADJ_PATH).astype(np.float32)          # (T,N,N) wind A(t), no self-loops
assert dyn_adj.shape == (T, N, N), (dyn_adj.shape, (T, N, N))

print(f'Stations   : {N}')
print(f'Timestamps : {T:,}  ({timestamps[0]} -> {timestamps[-1]})')
print(f'dyn_adj    : {dyn_adj.shape}  {dyn_adj.nbytes/1e6:.0f} MB')

# geometry for the forecast-climatology graph (same recipe as 03_dynamic_adjacency)
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi, dlmb = np.radians(lat2 - lat1), np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dlmb/2)**2
    return 2*R*np.arctan2(np.sqrt(a), np.sqrt(1-a))

dist_matrix = np.zeros((N, N))
for i in range(N):
    for j in range(N):
        if i != j:
            dist_matrix[i, j] = haversine_km(lats[i], lons[i], lats[j], lons[j])
within_threshold = (dist_matrix > 0) & (dist_matrix <= THRESHOLD_KM)
dir_u = np.zeros((N, N)); dir_v = np.zeros((N, N))
for i in range(N):
    for j in range(N):
        if i != j:
            de = (lons[j]-lons[i]) * np.cos(np.radians(lats[i]))
            dn = (lats[j]-lats[i])
            nrm = np.hypot(de, dn)
            if nrm > 0:
                dir_u[i, j], dir_v[i, j] = de/nrm, dn/nrm
print(f'Edges within {THRESHOLD_KM} km: {int(within_threshold.sum())}')

# HOW TO CHECK: 'Edges within 12 km' matches 03_dynamic_adjacency. dyn_adj ~105 MB.

Stations   : 37
Timestamps : 19,921  (2024-01-01 00:00:00 -> 2026-04-10 00:00:00)
dyn_adj    : (19921, 37, 37)  109 MB
Edges within 12.0 km: 508


In [4]:
BASE_FEATS = ['pm2.5','pm10','no2','so2','co','o3','rh','at','u_wind','v_wind']
g = df.groupby('Station', sort=False)

df['wspd'] = np.hypot(df['u_wind'], df['v_wind'])

# rolling PM2.5 stats - window ends at 'now' (an observed hour), so no leakage
df['pm_roll6_mean']  = g['pm2.5'].transform(lambda s: s.rolling(6,  min_periods=1).mean())
df['pm_roll6_std']   = g['pm2.5'].transform(lambda s: s.rolling(6,  min_periods=1).std()).fillna(0.0)
df['pm_roll24_mean'] = g['pm2.5'].transform(lambda s: s.rolling(24, min_periods=1).mean())
df['pm_roll24_std']  = g['pm2.5'].transform(lambda s: s.rolling(24, min_periods=1).std()).fillna(0.0)
df['pm_delta3']      = g['pm2.5'].transform(lambda s: s - s.shift(3)).fillna(0.0)

tsg = df.groupby('Timestamp', sort=False)['pm2.5']
df['city_mean_pm']  = tsg.transform('mean')
df['city_max_pm']   = tsg.transform('max')
df['pm_minus_city'] = df['pm2.5'] - df['city_mean_pm']

hh  = df['Timestamp'].dt.hour; dow = df['Timestamp'].dt.dayofweek; mon = df['Timestamp'].dt.month
df['hour_sin'] = np.sin(2*np.pi*hh/24);      df['hour_cos'] = np.cos(2*np.pi*hh/24)
df['dow_sin']  = np.sin(2*np.pi*dow/7);      df['dow_cos']  = np.cos(2*np.pi*dow/7)
df['mon_sin']  = np.sin(2*np.pi*(mon-1)/12); df['mon_cos']  = np.cos(2*np.pi*(mon-1)/12)
df['is_weekend'] = (dow >= 5).astype(float)

FEATURE_COLS = (BASE_FEATS +
    ['wspd','pm_roll6_mean','pm_roll6_std','pm_roll24_mean','pm_roll24_std','pm_delta3',
     'city_mean_pm','city_max_pm','pm_minus_city',
     'hour_sin','hour_cos','dow_sin','dow_cos','mon_sin','mon_cos','is_weekend'])
F = len(FEATURE_COLS)
TIME_COLS = ['hour_sin','hour_cos','dow_sin','dow_cos','mon_sin','mon_cos','is_weekend']
TIME_IDX  = [FEATURE_COLS.index(c) for c in TIME_COLS]

X_full = np.zeros((T, N, F), dtype=np.float32)
for fi, col in enumerate(FEATURE_COLS):
    wide = df.pivot_table(index='Timestamp', columns='Station', values=col).reindex(columns=station_names)
    X_full[:, :, fi] = np.nan_to_num(wide.values.astype(np.float32))
Y_full = X_full[:, :, 0].copy()                                  # PM2.5 target (channel 0)

print(f'F = {F} features')
print(FEATURE_COLS)
print(f'X_full {X_full.shape}  {X_full.nbytes/1e6:.0f} MB   NaNs: {int(np.isnan(X_full).sum())}')

# HOW TO CHECK: F == 26, X_full ~77 MB, NaNs == 0.

C:\Users\jlakk_j7eh5fd\AppData\Local\Temp\ipykernel_25660\382421343.py:18: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  hh  = df['Timestamp'].dt.hour; dow = df['Timestamp'].dt.dayofweek; mon = df['Timestamp'].dt.month


F = 26 features
['pm2.5', 'pm10', 'no2', 'so2', 'co', 'o3', 'rh', 'at', 'u_wind', 'v_wind', 'wspd', 'pm_roll6_mean', 'pm_roll6_std', 'pm_roll24_mean', 'pm_roll24_std', 'pm_delta3', 'city_mean_pm', 'city_max_pm', 'pm_minus_city', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'mon_sin', 'mon_cos', 'is_weekend']
X_full (19921, 37, 26)  77 MB   NaNs: 0


In [5]:
train_end_idx = int(T * 0.70)                                    # 13944
val_end_idx   = int(T * 0.85)                                    # 16932

def split_starts(lo, hi):
    n = (hi - lo) - (INPUT_WINDOW + HORIZON) + 1                  # windows fully inside [lo,hi)
    return np.arange(lo, lo + n)

train_starts = split_starts(0, train_end_idx)
val_starts   = split_starts(train_end_idx, val_end_idx)
test_starts  = split_starts(val_end_idx, T)
print(f'windows -- train {len(train_starts):,} | val {len(val_starts):,} | test {len(test_starts):,}')

train_ts_mask = np.zeros(T, bool); train_ts_mask[:train_end_idx] = True

feat_scaler = StandardScaler().fit(X_full[train_ts_mask].reshape(-1, F))
Xs_full = feat_scaler.transform(X_full.reshape(-1, F)).reshape(T, N, F).astype(np.float32)

pm_train   = Y_full[train_ts_mask].reshape(-1, 1)
log_scaler = StandardScaler().fit(np.log1p(np.clip(pm_train, 0, None)))
def to_log(a):
    return log_scaler.transform(np.log1p(np.clip(a, 0, None)).reshape(-1, 1)).reshape(a.shape).astype(np.float32)
def from_log(a):
    return np.expm1(log_scaler.inverse_transform(a.reshape(-1, 1)).reshape(a.shape))
Yl_full = to_log(Y_full)                                         # anchor = Yl_full[now] later

_std = Xs_full[train_ts_mask].reshape(-1, F).std(0)
print(f'raw PM2.5 skew {pd.Series(pm_train.ravel()).skew():.2f} -> log skew '
      f'{pd.Series(np.log1p(pm_train.ravel())).skew():.2f}')
print(f'per-feature std after scaling: min {_std.min():.2f}  max {_std.max():.2f}')

del X_full                                                       # free RAM - only Xs_full needed now

# HOW TO CHECK: window counts 13873 / 2917 / 2918 (same as 07). log skew ~0. std ~1.

windows -- train 13,873 | val 2,917 | test 2,918
raw PM2.5 skew 2.56 -> log skew -0.18
per-feature std after scaling: min 1.00  max 1.00


In [6]:
# forecast-horizon adjacency from a TRAIN-only wind climatology
def compute_At_vec(u, v, eps=1e-6):
    align = u[:, None]*dir_u + v[:, None]*dir_v                  # wind at i . direction(i->j)
    A = np.where(within_threshold & (align > 0.0), align/(dist_matrix + eps), 0.0).astype(np.float32)
    rs = A.sum(1, keepdims=True); rs[rs == 0] = 1.0
    return A / rs

trm = df['Timestamp'].isin(pd.DatetimeIndex(timestamps[:train_end_idx]))
cl = (df[trm].assign(mo=df.loc[trm, 'Timestamp'].dt.month - 1,
                     ho=df.loc[trm, 'Timestamp'].dt.hour)
             .groupby(['Station', 'mo', 'ho'])[['u_wind', 'v_wind']].mean())
st_mean = df[trm].groupby('Station')[['u_wind', 'v_wind']].mean()
cl_idx = set(cl.index)

clim_grid = np.zeros((12, 24, N, 2), dtype=np.float32)           # (month, hour, station, uv)
for si, s in enumerate(station_names):
    for mo in range(12):
        for ho in range(24):
            clim_grid[mo, ho, si] = (cl.loc[(s, mo, ho)].values
                                     if (s, mo, ho) in cl_idx else st_mean.loc[s].values)

mo_arr = pd.DatetimeIndex(timestamps).month.values - 1
ho_arr = pd.DatetimeIndex(timestamps).hour.values
t0 = time.time()
dyn_clim = np.zeros((T, N, N), dtype=np.float32)
for t in range(T):
    uv = clim_grid[mo_arr[t], ho_arr[t]]
    dyn_clim[t] = compute_At_vec(uv[:, 0], uv[:, 1])
print(f'dyn_clim {dyn_clim.shape} built in {time.time()-t0:.0f}s  ({dyn_clim.nbytes/1e6:.0f} MB)')
print(f'edges/hr -- hourly A(t) {np.mean((dyn_adj>0).sum((1,2))):.1f} | '
      f'climatology {np.mean((dyn_clim>0).sum((1,2))):.1f}')

import gc; del df, trm, cl; gc.collect()                         # free RAM - df no longer needed

# HOW TO CHECK: climatology edges/hr in the same ballpark as hourly (smoother). <60s.

dyn_clim (19921, 37, 37) built in 0s  (109 MB)
edges/hr -- hourly A(t) 259.6 | climatology 264.1


32

In [7]:
class EcoWindowDS(Dataset):
    """Slices the shared full-length arrays by window-start index - no windowed copies."""
    def __init__(self, starts, weights=None, transpose=False):
        self.s = np.asarray(starts, dtype=np.int64); self.tr = transpose
        self.w = (weights if weights is not None else np.ones(len(self.s), np.float32)).astype(np.float32)
    def _o(self, a):
        return np.ascontiguousarray(np.swapaxes(a, -1, -2)) if self.tr else a.copy()
    def __len__(self):
        return len(self.s)
    def __getitem__(self, k):
        gg = int(self.s[k]); i1 = gg + INPUT_WINDOW; i2 = i1 + HORIZON
        return (torch.from_numpy(Xs_full[gg:i1].copy()),           # (48,N,F)
                torch.from_numpy(self._o(dyn_adj[gg:i1])),           # (48,N,N)
                torch.from_numpy(self._o(dyn_clim[i1:i2])),          # (24,N,N)
                torch.from_numpy(Xs_full[i1:i2, 0, :][:, TIME_IDX].copy()),  # (24,n_time)
                torch.from_numpy(Yl_full[i1-1].copy()),            # (N,) last obs PM2.5 (log)
                torch.from_numpy(Yl_full[i1:i2].copy()),           # (24,N) target (log)
                torch.tensor(self.w[k], dtype=torch.float32))

def window_swing(starts):
    sw = np.empty(len(starts), np.float32)
    for k, gg in enumerate(starts):
        seg = Y_full[gg+INPUT_WINDOW : gg+INPUT_WINDOW+HORIZON]
        sw[k] = (seg.max(0) - seg.min(0)).mean()
    return sw

tr_swing = window_swing(train_starts)
med = float(np.median(tr_swing))
tr_w = 1.0 + LOSS_VOL_ALPHA * (tr_swing / (med + 1e-6))
tr_w = np.clip(tr_w, 0.5, 4.0).astype(np.float32)
tr_w *= len(tr_w) / tr_w.sum()
print(f'train swing: median {med:.1f}  p90 {np.percentile(tr_swing,90):.1f}')
print(f'loss weights: min {tr_w.min():.2f}  mean {tr_w.mean():.2f}  max {tr_w.max():.2f}')


val_ld_T   = DataLoader(EcoWindowDS(val_starts,  transpose=True),  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
train_ld_T = DataLoader(EcoWindowDS(train_starts, tr_w, transpose=True), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_ld_T  = DataLoader(EcoWindowDS(test_starts,  transpose=True),  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_ld_B  = DataLoader(EcoWindowDS(test_starts,  transpose=False), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f'batches -- train {len(train_ld_T)} | val {len(val_ld_T)} | test {len(test_ld_T)}')
# HOW TO CHECK: batches 868 / 183 / 183 (same counts as 09).

train swing: median 107.4  p90 224.5
loss weights: min 0.56  mean 1.00  max 1.87
batches -- train 868 | val 183 | test 183


In [8]:
class WATGCNv2(nn.Module):
    def __init__(self, in_features, hidden_dim, gru_layers, attn_heads,
                 forecast_horizon, n_stations, dropout=0.2, n_time=7, gcn_hops=2):
        super().__init__()
        H = hidden_dim
        self.input_proj = nn.Linear(in_features, H)
        self.gcn_lin  = nn.ModuleList(nn.Linear(H, H) for _ in range(gcn_hops))
        self.gcn_norm = nn.ModuleList(nn.LayerNorm(H) for _ in range(gcn_hops))
        self.act = nn.ReLU()
        self.gru = nn.GRU(H, H, gru_layers, batch_first=True,
                          dropout=dropout if gru_layers > 1 else 0.0)
        self.attn = nn.MultiheadAttention(H, attn_heads, dropout=dropout, batch_first=True)
        self.attn_norm = nn.LayerNorm(H)
        self.dropout = nn.Dropout(dropout)
        self.time_enc = nn.Sequential(nn.Linear(n_time, H), nn.ReLU(), nn.Linear(H, H))
        self.decoder = nn.Sequential(nn.Linear(H*3, H), nn.ReLU(), nn.Dropout(dropout), nn.Linear(H, 1))
        self.anchor_gate = nn.Parameter(torch.linspace(1.0, 0.3, forecast_horizon))
        self.H, self.FH = H, forecast_horizon

    @staticmethod
    def _norm_adj(A, n, device):
        I = torch.eye(n, device=device).unsqueeze(0).unsqueeze(0)
        A = A + I
        return A / A.sum(-1, keepdim=True).clamp(min=1e-6)

    def forward(self, X, At, Afc, TFut, Anchor):
        B, Tin, n, _ = X.shape
        h = self.act(self.input_proj(X))                          # (B,Tin,n,H)
        A = self._norm_adj(At, n, X.device)
        for lin, nrm in zip(self.gcn_lin, self.gcn_norm):
            h = self.act(nrm(lin(torch.matmul(A, h)))) + h        # residual graph hop
        hf = h.permute(0, 2, 1, 3).reshape(B*n, Tin, self.H)
        gg, _ = self.gru(hf)
        gg = self.dropout(gg)
        aa, _ = self.attn(gg, gg, gg)
        ctx = self.attn_norm(aa + gg)[:, -1, :].reshape(B, n, self.H)   # (B,n,H)
        Afc_n = self._norm_adj(Afc, n, X.device)                  # (B,FH,n,n)
        ctx_fc = torch.matmul(Afc_n, ctx.unsqueeze(1).expand(B, self.FH, n, self.H))  # (B,FH,n,H)
        t_emb = self.time_enc(TFut).unsqueeze(2).expand(B, self.FH, n, self.H)
        ctx_rep = ctx.unsqueeze(1).expand(B, self.FH, n, self.H)
        resid = self.decoder(torch.cat([ctx_rep, ctx_fc, t_emb], dim=-1)).squeeze(-1)  # (B,FH,n)
        anchor = Anchor.unsqueeze(1) * self.anchor_gate.view(1, self.FH, 1)
        return resid + anchor

_m = WATGCNv2(F, HIDDEN_DIM, GRU_LAYERS, ATTN_HEADS, HORIZON, N, DROPOUT, len(TIME_IDX), GCN_HOPS)
print(f'v2 parameters: {sum(p.numel() for p in _m.parameters()):,}')
del _m

v2 parameters: 18,201


In [9]:
def weighted_huber(pred, target, w, delta=1.0):
    absr = (pred - target).abs()
    quad = torch.clamp(absr, max=delta)
    per = (0.5*quad**2 + delta*(absr - quad)).mean(dim=(1, 2))    # (B,)
    return (per * w).mean() / w.mean().clamp(min=1e-6)

def run_epoch(model, loader, opt=None, sched=None):
    is_train = opt is not None
    model.train(is_train)
    tot, n = 0.0, 0
    for X, At, Afc, tf, anc, y, w in loader:
        X, At, Afc = X.to(DEVICE), At.to(DEVICE), Afc.to(DEVICE)
        tf, anc, y, w = tf.to(DEVICE), anc.to(DEVICE), y.to(DEVICE), w.to(DEVICE)
        with torch.set_grad_enabled(is_train):
            out = model(X, At, Afc, tf, anc)
            loss = weighted_huber(out, y, w if is_train else torch.ones_like(w))
        if is_train:
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            if sched is not None: sched.step()
        tot += loss.item() * X.size(0); n += X.size(0)
    return tot / n

@torch.no_grad()
def predict(model, loader):
    model.eval(); P, A = [], []
    for X, At, Afc, tf, anc, y, w in loader:
        out = model(X.to(DEVICE), At.to(DEVICE), Afc.to(DEVICE), tf.to(DEVICE), anc.to(DEVICE))
        P.append(out.cpu().numpy()); A.append(y.numpy())
    return np.concatenate(P), np.concatenate(A)

def score(pred_log, act_log):
    pr, ar = from_log(pred_log), from_log(act_log)
    return dict(rmse=float(np.sqrt(mean_squared_error(ar.ravel(), pr.ravel()))),
                mae=float(mean_absolute_error(ar.ravel(), pr.ravel())),
                r2=float(r2_score(ar.ravel(), pr.ravel())), preds=pr, actual=ar)

def train_model(seed, max_epochs=60, patience=12, lr=1e-3, verbose=True):
    torch.manual_seed(seed); np.random.seed(seed)
    model = WATGCNv2(F, HIDDEN_DIM, GRU_LAYERS, ATTN_HEADS, HORIZON, N,
                     DROPOUT, len(TIME_IDX), GCN_HOPS).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    steps = len(train_ld) * max_epochs
    warm = max(1, int(0.05 * steps))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: s/warm if s < warm else
        0.5*(1 + np.cos(np.pi*(s-warm)/max(1, steps-warm))))
    best, best_state, ctr = 1e9, None, 0
    hist = {'train': [], 'val': []}
    for ep in range(1, max_epochs+1):
        t0 = time.time()
        tr = run_epoch(model, train_ld, opt, sched)
        vl = run_epoch(model, val_ld)
        hist['train'].append(tr); hist['val'].append(vl)
        if vl < best - 1e-5:
            best, best_state, ctr = vl, copy.deepcopy(model.state_dict()), 0
            tag = 'best'
        else:
            ctr += 1; tag = f'{ctr}/{patience}'
        if verbose:
            print(f'  ep {ep:>2} | train {tr:.5f} | val {vl:.5f} | {time.time()-t0:>4.0f}s | {tag}')
        if ctr >= patience:
            break
    model.load_state_dict(best_state)
    return model, best, hist

print('train / predict / score / train_model ready.')

train / predict / score / train_model ready.


In [10]:
# 1) matched baseline: the saved v2 models (graph as built) for the same 5 seeds, scored on the test set
SEEDS5 = [42, 7, 123, 2024, 999]
base_models = []
for sd in SEEDS5:
    m = WATGCNv2(F, HIDDEN_DIM, GRU_LAYERS, ATTN_HEADS, HORIZON, N, DROPOUT, len(TIME_IDX), GCN_HOPS).to(DEVICE)
    m.load_state_dict(torch.load(os.path.join(MODEL_DIR, f'watgcn_v2_seed{sd}.pt'), map_location=DEVICE, weights_only=True))
    m.eval(); base_models.append(m)
base_preds, test_actual = [], None
for m in base_models:
    p, a = predict(m, test_ld_B); base_preds.append(p); test_actual = a
base_ens = score(np.mean(base_preds, axis=0), test_actual)
print(f"baseline (A as built), 5-seed ensemble: RMSE {base_ens['rmse']:.2f} | MAE {base_ens['mae']:.2f} | R2 {base_ens['r2']:.3f}")

# HOW TO CHECK: RMSE about 60.9 (the saved 7-seed v2 ensemble is 60.92; the 5-seed one is slightly different, ~60.9-61.4).

baseline (A as built), 5-seed ensemble: RMSE 60.88 | MAE 39.82 | R2 0.635


In [11]:
# 2) train the same model with A and A_fc TRANSPOSED (5 seeds, ~4-6 minutes each on the RTX 3050)
train_ld, val_ld = train_ld_T, val_ld_T          # train_model() reads these globals
T_models, T_val = [], []
for sd in SEEDS5:
    print(f'\n=== transposed graph, seed {sd} ===')
    m, b, h = train_model(sd)
    torch.save(m.state_dict(), os.path.join(MODEL_DIR, f'watgcn_v2T_seed{sd}.pt'))
    T_models.append(m)
print('done')

# HOW TO CHECK: val loss falls each epoch and early-stops around epoch 10-20 (best epoch ~4-7 like v2).


=== transposed graph, seed 42 ===
  ep  1 | train 0.20404 | val 0.15230 |   21s | best
  ep  2 | train 0.15141 | val 0.14618 |   20s | best
  ep  3 | train 0.13866 | val 0.14768 |   20s | 1/12
  ep  4 | train 0.13314 | val 0.14537 |   20s | best
  ep  5 | train 0.12801 | val 0.13191 |   20s | best
  ep  6 | train 0.12376 | val 0.14314 |   21s | 1/12
  ep  7 | train 0.11988 | val 0.13393 |   21s | 2/12
  ep  8 | train 0.11629 | val 0.13754 |   22s | 3/12
  ep  9 | train 0.11260 | val 0.13864 |   21s | 4/12
  ep 10 | train 0.10988 | val 0.14324 |   21s | 5/12
  ep 11 | train 0.10641 | val 0.14580 |   20s | 6/12
  ep 12 | train 0.10460 | val 0.14291 |   22s | 7/12
  ep 13 | train 0.10176 | val 0.14892 |   23s | 8/12
  ep 14 | train 0.09984 | val 0.14579 |   23s | 9/12
  ep 15 | train 0.09828 | val 0.14659 |   22s | 10/12
  ep 16 | train 0.09694 | val 0.15115 |   21s | 11/12
  ep 17 | train 0.09539 | val 0.15264 |   20s | 12/12

=== transposed graph, seed 7 ===
  ep  1 | train 0.20220 | v

In [12]:
# 3) compare on the held-out test set
T_preds = []
for m in T_models:
    p, a = predict(m, test_ld_T); T_preds.append(p)
T_ens = score(np.mean(T_preds, axis=0), test_actual)

def by_horizon(s):
    return [float(np.sqrt(np.mean((s['preds'][:, h, :] - s['actual'][:, h, :])**2))) for h in (0, 5, 11, 17, 23)]

diff = T_ens['rmse'] - base_ens['rmse']
print(f"{'':28}{'RMSE':>8}{'MAE':>8}{'R2':>8}   +1h   +6h  +12h  +18h  +24h")
for name, s in (('A as built (5 seeds)', base_ens), ('A transposed (5 seeds)', T_ens)):
    hz = by_horizon(s)
    print(f"{name:28}{s['rmse']:>8.2f}{s['mae']:>8.2f}{s['r2']:>8.3f}  " + ' '.join(f'{x:5.1f}' for x in hz))
print(f"\ntransposed - as built = {diff:+.2f} ug/m3 RMSE")
print('Rule of thumb from this project: differences under ~1 ug/m3 are within seed variation.')

json.dump({'as_built_5seed': {k: base_ens[k] for k in ('rmse','mae','r2')},
           'transposed_5seed': {k: T_ens[k] for k in ('rmse','mae','r2')},
           'by_horizon_as_built': by_horizon(base_ens), 'by_horizon_transposed': by_horizon(T_ens),
           'rmse_diff_transposed_minus_as_built': diff, 'seeds': SEEDS5},
          open(os.path.join(RESULTS_DIR, 'v2_direction_test.json'), 'w'), indent=2)
print('saved results/v2_direction_test.json')

# HOW TO READ: a clearly lower RMSE for the transposed graph (more than ~1 ug/m3) means the original
# orientation was hurting and the wind graph should be re-evaluated. Within ~1 means the orientation
# does not matter for accuracy and the earlier "no measurable gain" conclusion stands either way.

                                RMSE     MAE      R2   +1h   +6h  +12h  +18h  +24h
A as built (5 seeds)           60.88   39.82   0.635   41.5  56.5  62.8  66.1  66.6
A transposed (5 seeds)         62.29   40.34   0.618   41.6  58.2  64.5  67.4  67.9

transposed - as built = +1.41 ug/m3 RMSE
Rule of thumb from this project: differences under ~1 ug/m3 are within seed variation.
saved results/v2_direction_test.json
